# SciQLop Fluent (Declarative) API

The fluent API provides a concise, chainable way to build plot panels.
Instead of imperative step-by-step calls, you describe the panel layout in a single expression.

<div align="center">
<img src="https://github.com/SciQLop/SciQLop/raw/main/SciQLop/resources/icons/SciQLop.png"/>
</div>

## 1. Imperative vs Fluent — side by side

### Imperative style (what you already know)

In [ ]:
from SciQLop.user_api.plot import create_plot_panel, ScaleType
from SciQLop.user_api import TimeRange

p = create_plot_panel()
p.time_range = TimeRange("2015-11-18T02:14:30", "2015-11-18T03:34:00")

p.plot("speasy/cda/MMS/MMS1/FGM/MMS1_FGM_SRVY_L2/mms1_fgm_b_gse_srvy_l2")

plot, _ = p.plot("speasy/cda/MMS/MMS1/DIS/MMS1_FPI_FAST_L2_DIS_MOMS/mms1_dis_numberdensity_fast")
plot.y_scale_type = ScaleType.Logarithmic
plot.set_y_range(0.1, 100)

### Fluent style — same result, more concise

In [ ]:
from SciQLop.user_api.plot import fluent

panel = (fluent.new_panel()
    .plot("speasy/cda/MMS/MMS1/FGM/MMS1_FGM_SRVY_L2/mms1_fgm_b_gse_srvy_l2")
    .subplot()
        .plot("speasy/cda/MMS/MMS1/DIS/MMS1_FPI_FAST_L2_DIS_MOMS/mms1_dis_numberdensity_fast")
        .log_y()
        .y_range(0.1, 100)
    .time_range("2015-11-18T02:14:30", "2015-11-18T03:34:00"))

## 2. Building multi-subplot panels

Use `.subplot()` to start a new set of axes. Multiple `.plot()` calls without `.subplot()` overlay on the same axes.

In [ ]:
panel = (fluent.new_panel()
    # First subplot: magnetic field
    .plot("speasy/cda/MMS/MMS1/FGM/MMS1_FGM_SRVY_L2/mms1_fgm_b_gse_srvy_l2")
    .y_range(-75, 75)

    # Second subplot: velocity
    .subplot()
    .plot("speasy/cda/MMS/MMS1/DIS/MMS1_FPI_FAST_L2_DIS_MOMS/mms1_dis_bulkv_gse_fast")
    .y_range(-400, 400)

    # Third subplot: density (log scale)
    .subplot()
    .plot("speasy/cda/MMS/MMS1/DIS/MMS1_FPI_FAST_L2_DIS_MOMS/mms1_dis_numberdensity_fast")
    .log_y()
    .y_range(0.01, 100)

    .time_range("2015-11-18T02:14:30", "2015-11-18T03:34:00"))

## 3. Overlaying multiple products on the same axes

Call `.plot()` multiple times without `.subplot()` to overlay products.

In [ ]:
panel = (fluent.new_panel()
    # Overlay survey and burst density on the same axes
    .plot("speasy/cda/MMS/MMS1/DIS/MMS1_FPI_FAST_L2_DIS_MOMS/mms1_dis_numberdensity_fast")
    .plot("speasy/cda/MMS/MMS1/DIS/MMS1_FPI_BRST_L2_DIS_MOMS/mms1_dis_numberdensity_brst")
    .log_y()
    .y_range(0.01, 100)
    .time_range("2015-11-18T02:14:30", "2015-11-18T03:34:00"))

## 4. Accessing the underlying PlotPanel

The builder returns a `PanelBuilder`. Use `.panel` to get the underlying `PlotPanel` if you need imperative access.

In [ ]:
builder = (fluent.new_panel()
    .plot("speasy/cda/MMS/MMS1/FGM/MMS1_FGM_SRVY_L2/mms1_fgm_b_gse_srvy_l2")
    .time_range("2015-11-18T02:14:30", "2015-11-18T03:34:00"))

# Get the PlotPanel for imperative operations
p = builder.panel
p.save("/tmp/my_panel.png")

## 5. Wrapping an existing panel

Use `fluent.panel(name)` to get a builder for an already-open panel.

In [ ]:
# Add a subplot to an existing panel by its tab name
# (fluent.panel("Panel 1")
#     .subplot()
#     .plot("speasy/cda/MMS/MMS1/DIS/MMS1_FPI_FAST_L2_DIS_MOMS/mms1_dis_temppara_fast")
#     .log_y())

## API reference

| Method | Effect |
|--------|--------|
| `fluent.new_panel()` | Create a new panel |
| `fluent.panel(name)` | Wrap an existing panel |
| `.plot(product)` | Add a graph (same args as `PlotPanel.plot()`) |
| `.subplot()` | Start new axes |
| `.y_range(lo, hi)` | Set Y bounds on current axes |
| `.log_y()` / `.linear_y()` | Set Y scale type |
| `.time_range(start, stop)` | Set panel time range |
| `.panel` | Access the underlying `PlotPanel` |